In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/MCG+01-32-049_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/MCG+01-32-049_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     377   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        58   (2094, 964)   float32   


In [3]:
image_cut = image[52:172, 0:1890]
image_error_cut = image_error[52:172, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.00309
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049\MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/PSF/Real_seeing_MGC+01-32-049.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 120, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))

sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.1330749867078596
1.6441797529293292
0.8458781390160057
0.8420459810754362
35.10550574347173
25.258012424177487
0.5444381643776929
0.5541519125246646
1.565531107783728
1.593451042091355
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       4.630665e+01  8.910700e-02  5.687778e-05   0.000252   0.000061
Green      9.681572e+00  5.266100e+01  0.000000e+00   0.000000   0.000000
Purple     2.493663e+01  3.345865e+01  9.913413e+01   7.101011   1.000247
Orange     1.907515e+01  1.379125e+01  8.658137e-01  90.767414  17.497082
Chocolate  6.287635e-12  4.948083e-11  4.419518e-09   2.131322  81.502610


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.129147200838127
2.883300614373746
2.5210612425223484
2.5075396922412567
58.12835583399633
43.441931674414015
1.6107307273832552
1.6357363142173322
5.609221596164578
5.671055508433014
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.691133e+01  2.375100e-01  6.940648e-05   0.000256   0.000069
Green      1.590707e+01  5.188983e+01  0.000000e+00   0.000000   0.000000
Purple     2.687411e+01  3.398881e+01  9.895365e+01   7.225797   1.176547
Orange     2.030749e+01  1.388385e+01  1.046283e+00  90.393621  20.047847
Chocolate  7.729064e-12  5.381986e-11  6.204980e-09   2.380327  78.775537


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)
data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

3.0854854672411713
3.896805595389216
4.763698908714648
4.8945354561681125
70.53506833040787
54.54708309745028
3.101496680878551
3.1300417740000195
7.36195047098205
7.429699414058988
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       2.777868e+01  3.623488e+00  8.729221e-05   0.000261   0.000079
Green      2.278925e+01  4.741251e+01  0.000000e+00   0.000000   0.000000
Purple     2.840372e+01  3.438754e+01  9.870551e+01   7.220076   1.356182
Orange     2.102835e+01  1.457647e+01  1.294404e+00  87.863715  23.025314
Chocolate  1.001238e-11  5.877184e-11  9.691015e-09   4.915948  75.618425


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
real_errors_HB = np.sqrt(radial_profile_error_HB**2+(sky_err[2])**2+(0.04*radial_profile_HB)**2)
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.566251806342246
0.5650321070332929
0.35903588777409357
0.3586418287509968
6.578444116120371
7.1473138140924295
0.4761943247679585
0.4730342540351261
0.44816642923461336
0.4753331822846345
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.594277e+01  9.261113e-01  0.000000e+00   0.000000   0.000000
Green      2.437507e+01  4.869733e+01  1.385007e-17   0.000000   0.000000
Purple     2.379770e+01  3.559833e+01  9.725031e+01   4.553512   1.994231
Orange     1.588446e+01  1.477823e+01  2.749694e+00  95.438659  62.310973
Chocolate  3.605537e-15  3.048492e-14  6.105261e-12   0.007829  35.694797


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.0995326241745127
1.0973012741203063
1.0842957977580454
1.0831524938062904
11.88642211125045
12.763395088859795
1.4196671768267588
1.4101113013446096
1.6812193166726666
1.7599826314334415
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5
Blue       3.236305e+01  1.451671e+00  0.000000e+00   0.000000   0.000000
Green      2.648871e+01  4.823208e+01  2.624583e-08   0.000000   0.000000
Purple     2.477478e+01  3.562985e+01  9.695304e+01   4.597216   2.238578
Orange     1.637346e+01  1.468640e+01  3.046963e+00  95.393294  68.979325
Chocolate  4.241210e-15  3.259568e-14  7.936192e-12   0.009490  28.782097


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5810278357826553
1.5775999296579004
2.180145307452582
2.178827828532162
15.656926915723634
16.62338953873506
2.7971961248020807
2.7793771492413986
2.4542156726694735
2.563882852635262
                 Peak 1        Peak 2         Peak 3     Peak 4     Peak 5
Blue       2.758799e+01  5.871493e+00  1.942610e-293   0.000000   0.000000
Green      2.899831e+01  4.586306e+01   5.500449e-06   0.000000   0.000000
Purple     2.631114e+01  3.390001e+01   9.652379e+01   4.491303   2.319237
Orange     1.710257e+01  1.436544e+01   3.476203e+00  95.478829  71.357293
Chocolate  5.449699e-15  3.324085e-14   1.160203e-11   0.029868  26.323469


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.2215741372953492
0.27761600814742493
0.1612525068170445
0.1637485443955396
1.6362445840675504
1.7600080023863378
0.4606207207161489
0.461432159082823
0.29923179385406157
0.3971582593134677
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       22.963837   0.716483  8.233402e-06  1.242280e-09  2.120374e-10
Green       0.027634  25.499410  2.378040e-22  4.146811e-57  1.964530e-65
Purple     36.623489  40.465470  8.828746e+01  3.423634e+00  2.897562e+00
Orange     40.380921  33.315124  1.171104e+01  9.650686e+01  8.716292e+01
Chocolate   0.004119   0.003513  1.494390e-03  6.950244e-02  9.939517e+00


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.42960057848407857
0.5176052368267658
0.4720339950655826
0.4791216820769885
3.0195057578595725
3.207769451130445
1.3659798392110527
1.3680346953142544
1.206087029043569
1.482694999381906
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       20.083873   1.150290  9.922311e-06  1.278316e-09  2.438312e-10
Green       0.200279  23.018131  7.539221e-21  1.166585e-56  3.022818e-64
Purple     38.019520  41.669074  8.729117e+01  3.469190e+00  2.973524e+00
Orange     41.692072  34.158902  1.270719e+01  9.645892e+01  8.934479e+01
Chocolate   0.004255   0.003604  1.624328e-03  7.189455e-02  7.681691e+00


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6255567847750833
0.7244910252035761
0.8784953407150351
0.9180187786836149
4.093845320429202
4.299052554309666
2.616347503505755
2.6189370062478687
1.788703652238779
2.0974310147349016
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       16.350447   5.253870  1.304468e-05  1.266352e-09  2.584045e-10
Green       1.047343  15.561259  4.160147e-19  6.104525e-56  1.725964e-63
Purple     39.584989  43.142440  8.591182e+01  3.520486e+00  3.019596e+00
Orange     43.012826  36.038639  1.408637e+01  9.638366e+01  9.076448e+01
Chocolate   0.004394   0.003792  1.805873e-03  9.585069e-02  6.215928e+00


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.33685530582090284
0.33618505412095456
0.21958610751729077
0.2194990816213494
2.008266576609512
2.1324086571209246
0.47914181561713876
0.4800372280215855
0.43652589817917736
0.43723387063110897
              Peak 1     Peak 2         Peak 3        Peak 4     Peak 5
Blue       21.236121   5.460762   8.595847e-95  0.000000e+00   0.000000
Green      13.862948  25.382899  2.958005e-218  0.000000e+00   0.000000
Purple     29.758979  37.464511   8.842385e+01  3.771975e+00   2.207753
Orange     35.141952  31.691828   1.157615e+01  9.622802e+01  65.080319
Chocolate   0.000000   0.000000   0.000000e+00  5.109054e-07  32.711928


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

0.6689951629148767
0.6677553942686696
0.6616003198252537
0.6613669453519618
3.7412611452245064
3.9277407613669064
1.4222731342327266
1.4245514990913848
1.6933034955809687
1.6946862407980843
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       19.834485   5.827791  1.375999e-66   0.000000   0.000000
Green      14.576392  25.110836  7.626157e-59   0.000000   0.000000
Purple     30.173938  37.493598  8.756023e+01   3.818967   2.358685
Orange     35.415185  31.567776  1.243977e+01  96.180996  69.081735
Chocolate   0.000000   0.000000  0.000000e+00   0.000037  28.559580


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

0.991064272836994
0.9894789327771728
1.3161303064471015
1.315787277630387
5.104018602708134
5.3050883000379345
2.73294489033244
2.735797313464864
2.4131620000809266
2.414890911083316
              Peak 1     Peak 2        Peak 3     Peak 4     Peak 5
Blue       17.876405   8.466391  1.188730e-45   0.000000   0.000000
Green      15.317158  24.036069  2.967896e-16   0.000000   0.000000
Purple     30.903990  36.279044  8.630061e+01   3.858566   2.493525
Orange     35.902447  31.218496  1.369939e+01  96.133347  72.985251
Chocolate   0.000000   0.000000  0.000000e+00   0.008087  24.521224


## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/MGC+01-32-049/MCG+01-32-049_1/FORS2.2022-01-29T07_05_26.457/Fit/5_peaks/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5582164039191169
1.5029057348986714
1.0108496023075988
1.0102490044256802
24.749063621013423
27.284980148124674
0.5818969008686595
0.5782012625148907
0.6814577450737419
0.6743725826931537
              Peak 1     Peak 2        Peak 3         Peak 4        Peak 5
Blue       64.401128   5.664470  1.260743e-03   2.563081e-06  1.684040e-07
Green      15.394060  64.936908  3.208223e-10   0.000000e+00  0.000000e+00
Purple     12.891077  22.754600  9.907724e+01   3.948576e+00  1.035275e+00
Orange      7.313735   6.644021  9.214950e-01   9.605142e+01  5.095301e+01
Chocolate   0.000000   0.000000  0.000000e+00  1.820467e-151  4.801171e+01


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.9553067669751347
2.8723263823163636
3.032536940075861
3.0302976726097635
44.212824572384896
48.119411854707984
1.7350349411671635
1.7238634930021366
2.56846355176283
2.5329898844999823
              Peak 1     Peak 2        Peak 3        Peak 4        Peak 5
Blue       57.918772   6.201352  1.515636e-03  2.634225e-06  2.144058e-07
Green      20.533805  64.126849  2.200325e-07  0.000000e+00  0.000000e+00
Purple     13.829039  23.025014  9.896571e+01  3.994777e+00  1.179340e+00
Orange      7.718385   6.646786  1.032773e+00  9.600522e+01  5.609841e+01
Chocolate   0.000000   0.000000  0.000000e+00  2.137515e-09  4.272225e+01


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.215240094413153
4.123153189596946
5.93220921455872
5.908914810749884
57.226013559690294
61.52985753604896
3.4185167195186716
3.397736345454729
3.5694052298097576
3.5223503683407724
              Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       50.166577  12.425071   0.001972   0.000003  2.467524e-07
Green      26.740396  58.466584   0.000009   0.000000  0.000000e+00
Purple     14.965633  22.425194  98.798969   3.874291  1.290994e+00
Orange      8.127395   6.683151   1.199050  96.121203  6.094340e+01
Chocolate   0.000000   0.000000   0.000000   0.004504  3.776561e+01
